# pySCENIC — Mouse intestinal fibroblasts

This notebook infers and analyzes transcription-factor regulons in mouse
fibroblasts using the pySCENIC workflow:

**GRNBoost2 → cisTarget motif pruning → AUCell → regulon-level analysis**

The public notebook preserves the analysis method but does not hard-code
unpublished TF/regulon findings.


## Expected inputs and outputs

**Inputs**
- fibroblast raw-count `.h5ad`;
- species-specific TF list;
- 10 kb and 500 bp gene-based cisTarget ranking databases;
- species-specific v10 motif-to-TF annotations;
- a metadata column defining the groups used for regulon specificity analysis.

**Outputs**
- GRNBoost2 TF-target adjacency table;
- motif-supported regulons;
- AUCell-scored AnnData;
- cell-by-regulon AUC matrix;
- regulon UMAP;
- regulon-target table;
- mean activity and RSS tables/heatmaps;
- optional regulon overlays on the original Seurat UMAP.


## Function reference

| Function | Purpose |
|---|---|
| `load_pyscenic_config()` | Loads shared human/mouse pySCENIC settings. |
| `resolve_species_paths()` | Builds project-relative input, resource, and output paths. |
| `validate_pyscenic_inputs()` | Checks required files and metadata before long runs. |
| `tf_overlap()` | Checks how many TF-list genes occur in the expression matrix. |
| `extract_auc_matrix()` | Extracts the cell × regulon AUCell matrix from pySCENIC output. |
| `run_regulon_umap()` | Builds a UMAP from regulon activity rather than gene expression. |
| `mean_regulon_activity()` | Calculates average AUCell activity for every group. |
| `calculate_rss()` | Calculates regulon specificity scores for the metadata groups. |
| `top_rss_regulons()` | Returns the highest-RSS regulons per group. |
| `export_regulon_targets()` | Converts final regulons into a TF-target edge table. |
| `plot_embedding_by_group()` | Colors the SCENIC UMAP by a metadata group. |
| `plot_regulon_on_embedding()` | Colors an embedding by one regulon's AUCell activity. |
| `plot_rss_heatmap()` | Visualizes the highest-RSS regulons across groups. |
| `plot_mean_auc_heatmap()` | Visualizes mean AUCell activity for selected regulons. |
| `load_original_umap()` | Loads the original UMAP exported from Seurat. |


In [ ]:
from pathlib import Path

import anndata as ad
import pandas as pd

from python.pyscenic.pyscenic_utils import (
    save_run_metadata,
    load_pyscenic_config,
    resolve_species_paths,
    validate_pyscenic_inputs,
    tf_overlap,
    extract_auc_matrix,
    run_regulon_umap,
    mean_regulon_activity,
    calculate_rss,
    top_rss_regulons,
    export_regulon_targets,
    plot_embedding_by_group,
    plot_regulon_on_embedding,
    plot_rss_heatmap,
    plot_mean_auc_heatmap,
    load_original_umap,
)


## Load configuration


In [ ]:
PROJECT_DIR = Path(".")
SPECIES = "mouse"

CONFIG = load_pyscenic_config(
    PROJECT_DIR / "config" / "pyscenic_config.json"
)

PATHS = resolve_species_paths(
    CONFIG,
    species=SPECIES,
    project_dir=PROJECT_DIR,
)

GROUP_COL = CONFIG["group_col"]
THREADS = CONFIG["threads"]
SEED = CONFIG["seed"]


## Validate inputs

Checks that the fibroblast `.h5ad`, TF list, both cisTarget ranking databases,
motif annotations, and group metadata are available before starting the
computationally expensive steps.


In [ ]:
validate_pyscenic_inputs(
    PATHS,
    group_col=GROUP_COL,
)


## Inspect the raw-count AnnData and TF overlap

Confirms the basic matrix dimensions and checks that transcription factors in
the supplied Aerts-lab TF list are represented in the expression matrix.


In [ ]:
adata = ad.read_h5ad(PATHS["expression"])

print(adata)
print("Cells:", adata.n_obs)
print("Genes:", adata.n_vars)
print("Gene names unique:", adata.var_names.is_unique)

overlap_info = tf_overlap(
    adata,
    PATHS["tf_list"],
)

print({
    key: value
    for key, value in overlap_info.items()
    if key != "overlapping_tfs"
})


## Run GRNBoost2

Infers candidate TF→target co-expression relationships from the raw-count
expression matrix. This is the first, expression-based stage of pySCENIC;
motif evidence is added in the next step.


In [ ]:
adjacencies = PATHS["tables"] if "tables" in PATHS else PATHS["results"] / "tables"
adjacencies = PATHS["results"] / "tables" / "mouse_adjacencies.tsv"
grn_log = PATHS["results"] / "logs" / "grn.log"

!pyscenic grn \
    "{PATHS['expression']}" \
    "{PATHS['tf_list']}" \
    --method grnboost2 \
    --output "{adjacencies}" \
    --num_workers "{THREADS}" \
    --seed "{SEED}" \
    2>&1 | tee "{grn_log}"


In [ ]:
adj = pd.read_csv(adjacencies, sep="\t")
print(adj.shape)
display(adj.head())


## Run cisTarget motif pruning

Tests candidate GRNBoost2 modules against the 10 kb and 500 bp cisTarget
ranking databases. TF-target relationships are retained when supported by
cis-regulatory motif enrichment, producing the final regulons.

The main downstream analysis below uses the **unmasked** regulon set, matching
the final exploratory workflow. A masked run can be kept as a sensitivity check.


In [ ]:
regulon_dir = PATHS["results"] / "objects"
regulon_dir.mkdir(parents=True, exist_ok=True)

regulons_unmasked = regulon_dir / "mouse_regulons.csv"
ctx_log = PATHS["results"] / "logs" / "ctx.log"

!pyscenic ctx \
    "{adjacencies}" \
    "{PATHS['db_10kb']}" \
    "{PATHS['db_500bp']}" \
    --annotations_fname "{PATHS['motif_annotations']}" \
    --expression_mtx_fname "{PATHS['expression']}" \
    --output "{regulons_unmasked}" \
    --num_workers "{THREADS}" \
    --mode dask_multiprocessing \
    2>&1 | tee "{ctx_log}"


## Optional dropout-masked cisTarget run

This reproduces the additional `--mask_dropouts` sensitivity analysis from the
exploratory notebooks. It is separate from the main downstream regulon analysis.


In [ ]:
# regulons_masked = regulon_dir / "mouse_regulons_masked.csv"
# ctx_masked_log = PATHS["results"] / "logs" / "ctx_masked.log"
#
# !pyscenic ctx \
#     "{adjacencies}" \
#     "{PATHS['db_10kb']}" \
#     "{PATHS['db_500bp']}" \
#     --annotations_fname "{PATHS['motif_annotations']}" \
#     --expression_mtx_fname "{PATHS['expression']}" \
#     --output "{regulons_masked}" \
#     --num_workers "{THREADS}" \
#     --mode dask_multiprocessing \
#     --mask_dropouts \
#     2>&1 | tee "{ctx_masked_log}"


## Inspect AUCell ranking depth

Summarizes the number of detected genes per cell so the AUCell ranking depth can
be interpreted relative to the sparsity of the single-cell count matrix.


In [ ]:
import numpy as np
from scipy import sparse

if sparse.issparse(adata.X):
    genes_detected = np.asarray((adata.X > 0).sum(axis=1)).ravel()
else:
    genes_detected = np.sum(adata.X > 0, axis=1)

genes_detected = pd.Series(
    genes_detected,
    index=adata.obs_names,
    name="n_genes_detected",
)

display(
    genes_detected.quantile(
        [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 1.00]
    )
)


## Run AUCell

Scores each final regulon in every cell based on whether its target genes are
enriched near the top of that cell's expression ranking. The output is a
cell-by-regulon activity representation.


In [ ]:
scenic_output = (
    PATHS["results"]
    / "objects"
    / "mouse_scenic_unmasked.h5ad"
)
aucell_log = PATHS["results"] / "logs" / "aucell.log"

!pyscenic aucell \
    "{PATHS['expression']}" \
    "{regulons_unmasked}" \
    --output "{scenic_output}" \
    --num_workers "{THREADS}" \
    --seed "{SEED}" \
    2>&1 | tee "{aucell_log}"


## Extract the regulon AUC matrix

Reads the AUCell-scored AnnData and extracts the per-cell regulon activity
scores for downstream visualization and group-specific analysis.


In [ ]:
scenic_adata = ad.read_h5ad(scenic_output)

auc = extract_auc_matrix(
    scenic_adata
)

groups = scenic_adata.obs.loc[
    auc.index,
    GROUP_COL,
]

auc.to_csv(
    PATHS["results"] / "tables" / "regulon_auc_matrix.csv"
)

display(auc.iloc[:5, :5])


## Build a SCENIC regulon-activity UMAP

Creates a new UMAP using the AUCell activity matrix rather than gene expression.
Cells that share similar TF-regulon activity profiles should appear close in
this embedding.


In [ ]:
umap_cfg = CONFIG["umap"]

scenic_umap = run_regulon_umap(
    auc,
    n_neighbors=umap_cfg["n_neighbors"],
    min_dist=umap_cfg["min_dist"],
    metric=umap_cfg["metric"],
    random_state=SEED,
)

scenic_umap.to_csv(
    PATHS["results"] / "tables" / "regulon_activity_umap.csv"
)

plot_embedding_by_group(
    scenic_umap,
    groups,
    title=f"{CONFIG[SPECIES]['species_name']} regulon-activity UMAP",
    save_path=PATHS["results"] / "figures" / "regulon_activity_umap.png",
)


## Export final regulons and TF-target relationships

Converts pySCENIC's regulon file into a simple edge table containing regulon,
TF, target gene, and target weight. This is useful for target inspection and
human–mouse regulatory-program comparisons.


In [ ]:
regulon_targets = export_regulon_targets(
    regulons_unmasked,
    PATHS["results"] / "tables" / "regulon_targets.csv",
)

display(regulon_targets.head())


## Mean regulon activity by group

Calculates the average AUCell score of every regulon within each fibroblast
group. This answers which regulons have high average activity in each group.


In [ ]:
mean_auc = mean_regulon_activity(
    auc,
    groups,
)

mean_auc.to_csv(
    PATHS["results"] / "tables" / "mean_regulon_activity_by_group.csv"
)

display(mean_auc)


## Regulon specificity scores (RSS)

RSS emphasizes regulons whose activity is concentrated in one group rather
than simply having a high mean score everywhere. Invariant regulons are removed
before RSS is calculated.


In [ ]:
rss = calculate_rss(
    auc,
    groups,
)

rss.to_csv(
    PATHS["results"] / "tables" / "regulon_specificity_scores.csv"
)

top_rss = top_rss_regulons(
    rss,
    top_n=15,
)

top_rss.to_csv(
    PATHS["results"] / "tables" / "top_rss_regulons.csv",
    index=False,
)

display(top_rss)


## RSS and mean-AUCell heatmaps

The RSS heatmap highlights group-specific regulatory programs. The mean-AUCell
heatmap shows the actual average activity of the same selected regulons.


In [ ]:
_, _, rss_heatmap_data = plot_rss_heatmap(
    rss,
    top_n=15,
    save_path=PATHS["results"] / "figures" / "rss_heatmap.png",
)

selected_regulons = list(rss_heatmap_data.columns)

plot_mean_auc_heatmap(
    mean_auc,
    selected_regulons,
    save_path=PATHS["results"] / "figures" / "mean_auc_heatmap.png",
)


## Plot a selected regulon on the SCENIC UMAP

Supply a regulon name at runtime to visualize its AUCell activity without
hard-coding unpublished TF findings into the public notebook.


In [ ]:
# regulon_of_interest = "Regulon(TF_OF_INTEREST(+))"
#
# plot_regulon_on_embedding(
#     scenic_umap,
#     auc,
#     regulon_of_interest,
#     save_path=(
#         PATHS["results"]
#         / "figures"
#         / "selected_regulon_scenic_umap.png"
#     ),
# )


## Plot regulon activity on the original Seurat UMAP

Uses the original Seurat cell coordinates exported before pySCENIC and colors
those same cells by AUCell score. This lets regulon activity be interpreted in
the original fibroblast expression-space embedding.


In [ ]:
# original_umap = load_original_umap(
#     PROJECT_DIR
#     / "outputs"
#     / "pyscenic_exports"
#     / "mouse"
#     / "umap.tsv"
# )
#
# regulon_of_interest = "Regulon(TF_OF_INTEREST(+))"
#
# plot_regulon_on_embedding(
#     original_umap,
#     auc,
#     regulon_of_interest,
#     save_path=(
#         PATHS["results"]
#         / "figures"
#         / "selected_regulon_original_umap.png"
#     ),
# )


## Save run metadata

Records the resources, analysis parameters, and installed package versions used for this run.


In [ ]:
save_run_metadata(
    PATHS["metadata"] / "run_metadata.json",
    parameters={
        "threads": THREADS,
        "seed": SEED,
        "group_col": GROUP_COL,
        "auc_threshold": CONFIG["auc_threshold"],
        "umap": CONFIG["umap"],
    },
    resources={
        "tf_list": str(PATHS["tf_list"]),
        "db_10kb": str(PATHS["db_10kb"]),
        "db_500bp": str(PATHS["db_500bp"]),
        "motif_annotations": str(PATHS["motif_annotations"]),
    },
    extra={
        "species": SPECIES,
        "expression_file": str(PATHS["expression"]),
    },
)
